# Mission 2 Stage 4-A — 59D vs eGeMAPSv02 88D

질문: 59D handcrafted representation과 openSMILE eGeMAPSv02 Functionals 88D를 동일한 M04-100 조건에서 비교하면 internal validation 성능과 오류 구조가 어떻게 달라지는가?

한 번에 하나의 Checkpoint 셀만 직접 실행한다. 이 Notebook에는 저장된 실행 결과가 없다. 결과 해석과 다음 실험 결정은 사용자가 한다. 모델 입력은 현재 발화 waveform에서 추출한 feature뿐이다. record_key, call_id, speaker, crop 위치, duration 및 overlap 정보는 정렬·split·진단에만 사용한다.


## Checkpoint A — 환경과 고정 비교 계약

현재 브랜치와 소스 계약을 확인한다. 이 셀은 feature 추출이나 모델 학습을 시작하지 않는다.


In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys
import warnings

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src" / "dcc_data").is_dir():
    raise RuntimeError("STOP: repository root could not be resolved")
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

from dcc_data.audio import read_wav_frames
from dcc_data.mission2_model_experiments import (
    EXPECTED_INTERNAL_SPLIT, EXPECTED_OFFICIAL_CALLS, EXPECTED_OFFICIAL_ROWS,
    M04_MLP_CONFIG, build_standalone_diagnostic_report,
    calculate_paired_prediction_changes, calculate_prediction_metrics,
    create_fixed_internal_split, load_standalone_e05_cache,
    record_key_signature, standalone_group_split_indices,
    standalone_training_cache_paths,
)
from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID, CANONICAL_52D_FEATURE_NAMES, CANONICAL_52D_PREPARED_ID,
    M04_ITER100_FEATURE_SCREENING_CONFIG, NEXT_CANDIDATE_BLOCK_IDS,
    ROUND1_BLOCK_IDS, ROUND2_BLOCK_IDS, RESIDUAL_TARGET_BLOCK_IDS,
    PreparedRepresentation, _prepare_representation, _run_prepared_representation,
    load_candidate_feature_cache, prepare_residual_combination_screening,
)
from dcc_data.mission2_standard_representation import (
    REPRESENTATION_ID, build_egemaps_cache, extract_egemaps,
    get_egemaps_feature_names, load_egemaps_cache,
)

RAW_DATA_ROOT = REPO_ROOT / "data" / "raw_original"
MANIFEST_PATH = REPO_ROOT / "tmp" / "mission2_eda" / "utterance_manifest_v1.parquet"
CANDIDATE_CACHE_ROOT = REPO_ROOT / "data" / "processed" / "mission2_acoustic_feature_expansion"
EGEMAPS_CACHE_PATH = (
    REPO_ROOT / "data" / "processed" / "mission2_standard_representation"
    / "training_egemaps_v02_88d.npz"
)
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True
).strip()
print("Repository:", REPO_ROOT)
print("Branch:", branch)
print("59D reference: AF10_AF11_COMBINATION; classifier: M04-100")
print("Classifier config:", dict(M04_ITER100_FEATURE_SCREENING_CONFIG))
print("Expected internal split:", dict(EXPECTED_INTERNAL_SPLIT))


## Checkpoint B — 59D reference와 source provenance

기존 Training cache만 읽어 canonical 행 순서의 52D를 재구성하고, 기존 AF10+AF11 결합 helper로 59D를 준비한다. cache나 manifest가 없거나 key·call·label·split이 다르면 중단한다. 이 단계는 모델을 fit하지 않는다.


In [ ]:
e05_cache = load_standalone_e05_cache(
    standalone_training_cache_paths(REPO_ROOT),
    split="Training",
    expected_rows=EXPECTED_OFFICIAL_ROWS["Training"],
    expected_calls=EXPECTED_OFFICIAL_CALLS["Training"],
)
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("STOP: Training utterance manifest is missing")
manifest = pd.read_parquet(MANIFEST_PATH)
required_manifest = {
    "split", "record_key", "call_id", "speaker", "audio_path",
    "crop_start_frame", "crop_end_frame", "crop_ok",
}
if not required_manifest.issubset(manifest.columns):
    raise ValueError("STOP: manifest lacks required provenance/crop columns")
training_manifest = manifest.loc[manifest["split"].astype(str) == "Training"].copy()
keys = np.asarray(e05_cache.validated.record_keys).astype(str)
labels = np.asarray(e05_cache.validated.y, dtype=np.int64)
calls = e05_cache.tracking["call_id"].astype(str).to_numpy()
if len(training_manifest) != len(keys) or not training_manifest["record_key"].is_unique:
    raise ValueError("STOP: Training manifest row/key contract differs")
if set(training_manifest["record_key"].astype(str)) != set(keys):
    raise ValueError("STOP: Training manifest record_key set differs")
aligned_manifest = (
    training_manifest.set_index("record_key", verify_integrity=True)
    .loc[keys].reset_index()
)
if not np.array_equal(aligned_manifest["call_id"].astype(str).to_numpy(), calls):
    raise ValueError("STOP: manifest call_id alignment differs")
if not np.array_equal(aligned_manifest["speaker"].to_numpy(dtype=np.int64), labels):
    raise ValueError("STOP: manifest labels differ from frozen E05")
if aligned_manifest["audio_path"].isna().any() or aligned_manifest["audio_path"].astype(str).eq("").any():
    raise ValueError("STOP: manifest audio_path is missing")
if not aligned_manifest["crop_ok"].astype(bool).all():
    raise ValueError("STOP: manifest has invalid crops; no rows may be dropped")
train_idx, validation_idx, split_audit = create_fixed_internal_split(
    calls, split_function=standalone_group_split_indices,
    expected_contract=EXPECTED_INTERNAL_SPLIT,
)

cache_specs = (
    ("AF02", ROUND1_BLOCK_IDS), ("AF04", ROUND2_BLOCK_IDS),
    ("AF09", NEXT_CANDIDATE_BLOCK_IDS),
    ("AF10/11/12", RESIDUAL_TARGET_BLOCK_IDS),
)
feature_caches = {}
for label, block_ids in cache_specs:
    frame = load_candidate_feature_cache(
        manifest=manifest, cache_root=CANDIDATE_CACHE_ROOT,
        split="Training", block_ids=block_ids,
    )
    if not np.array_equal(frame["record_key"].astype(str).to_numpy(), keys):
        raise ValueError(f"STOP: {label} cache record_key order differs")
    if not np.array_equal(frame["call_id"].astype(str).to_numpy(), calls):
        raise ValueError(f"STOP: {label} cache call_id order differs")
    feature_caches[label] = frame

af02_names = BLOCK_BY_ID["AF02_PITCH_VOICING"].feature_names
af04_names = BLOCK_BY_ID["AF04_CPP"].feature_names
af09_names = BLOCK_BY_ID["AF09_MULTIBAND_ENVELOPE_COHERENCE"].feature_names
X_52d = np.concatenate((
    e05_cache.validated.X,
    feature_caches["AF02"].loc[:, af02_names].to_numpy(dtype=np.float32),
    feature_caches["AF04"].loc[:, af04_names].to_numpy(dtype=np.float32),
    feature_caches["AF09"].loc[:, af09_names].to_numpy(dtype=np.float32),
), axis=1).astype(np.float32, copy=False)
prepared_52d = _prepare_representation(
    representation_id=CANONICAL_52D_PREPARED_ID, X=X_52d, y=labels,
    record_keys=keys, feature_names=CANONICAL_52D_FEATURE_NAMES,
    train_idx=train_idx, validation_idx=validation_idx,
    split_audit=split_audit,
)
prepared_59d = prepare_residual_combination_screening(
    prepared_52d=prepared_52d, X_52d=X_52d, y=labels,
    record_keys=keys, call_ids=calls,
    candidate_frame=feature_caches["AF10/11/12"],
    canonical_train_idx=train_idx, canonical_validation_idx=validation_idx,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
)["AF10_AF11_COMBINATION"]
source_provenance = (
    "Training|" + hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
    + "|" + record_key_signature(keys)
)
print("59D rows/features:", len(keys), len(prepared_59d.feature_names))
print("Calls / train / validation / overlap:",
      len(np.unique(calls)), len(train_idx), len(validation_idx),
      split_audit["call_overlap"])
print("Manifest/key provenance:", source_provenance[:75] + "...")
display(pd.DataFrame([dict(e05_cache.cache_audit)]))
display(pd.DataFrame([split_audit]))


## Checkpoint C — 소수 발화와 synthetic 입력 호환성

실제 발화 소수만 읽기 전용으로 확인한다. 짧은 synthetic 신호는 NaN 대체 없이 오류가 나야 하며, 무음의 88D 값은 유한해야 한다. 8 kHz는 그대로 사용한다. 8 kHz Nyquist가 4 kHz이므로 openSMILE의 더 높은 대역 정의는 전체 실험 전 해석상 검토가 필요하다.


In [ ]:
feature_names_88 = get_egemaps_feature_names()
if len(feature_names_88) != 88 or len(set(feature_names_88)) != 88:
    raise ValueError("STOP: eGeMAPSv02 feature name contract differs")
sample_rows = aligned_manifest.loc[(aligned_manifest["crop_end_frame"] - aligned_manifest["crop_start_frame"]) >= 8_000].head(3)
if len(sample_rows) != 3:
    raise ValueError("STOP: compatibility sample is incomplete")
for row in sample_rows.itertuples():
    audio = read_wav_frames(RAW_DATA_ROOT / str(row.audio_path))
    if audio.sample_rate != 8_000 or audio.waveform.shape[0] != 1:
        raise ValueError("STOP: expected native 8 kHz mono WAV")
    crop = audio.waveform[
        0, int(row.crop_start_frame):int(row.crop_end_frame)
    ]
    values, names = extract_egemaps(crop, audio.sample_rate)
    if names != feature_names_88 or values.shape != (88,) or not np.isfinite(values).all():
        raise ValueError("STOP: small real utterance eGeMAPS contract failed")
    print("record_key / samples / dim:", row.record_key, len(crop), len(values))
silence, _ = extract_egemaps(np.zeros(8_000, dtype=np.float32), 8_000)
if not np.isfinite(silence).all():
    raise ValueError("STOP: silence produced NaN or inf")
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Segment too short")
    try:
        extract_egemaps(np.zeros(80, dtype=np.float32), 8_000)
    except ValueError as error:
        print("Expected very-short failure:", error)
    else:
        raise ValueError("STOP: very-short input was silently accepted")


## Checkpoint D — 전체 Training 88D cache 생성 또는 검증 로드

이 셀을 직접 실행할 때만 전체 추출이 시작된다. 기존 cache가 있으면 strict load만 한다. 없으면 WAV를 call 단위로 읽고 현재 발화 crop만 추출해, 모든 행을 채운 뒤 새 cache를 저장한다. 매우 짧은 발화에서 NaN이 나오면 실패하며 행 삭제·0 대체는 하지 않는다. 889,358×88 float32 행렬 자체가 약 313 MB이고 추가 메모리가 필요하다.


In [ ]:
if EGEMAPS_CACHE_PATH.is_file():
    egemaps_data = load_egemaps_cache(
        EGEMAPS_CACHE_PATH, keys, calls,
        source_provenance=source_provenance,
    )
else:
    matrix_88 = np.full((len(keys), 88), np.nan, dtype=np.float32)
    groups = aligned_manifest.groupby("audio_path", sort=False)
    for call_number, (audio_path, group) in enumerate(groups, start=1):
        audio = read_wav_frames(RAW_DATA_ROOT / str(audio_path))
        if audio.sample_rate != 8_000 or audio.waveform.shape[0] != 1:
            raise ValueError(f"STOP: unexpected WAV contract: {audio_path}")
        for row in group.itertuples():
            start, end = int(row.crop_start_frame), int(row.crop_end_frame)
            if start < 0 or end <= start or end > audio.waveform.shape[1]:
                raise ValueError(f"STOP: invalid crop for {row.record_key}")
            vector, names = extract_egemaps(audio.waveform[0, start:end], 8_000)
            if names != feature_names_88:
                raise ValueError("STOP: eGeMAPSv02 feature order changed")
            matrix_88[row.Index] = vector
        if call_number % 500 == 0:
            print("Calls extracted:", call_number)
    if not np.isfinite(matrix_88).all():
        raise ValueError("STOP: complete 88D matrix contains NaN or inf")
    EGEMAPS_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    egemaps_data = build_egemaps_cache(
        EGEMAPS_CACHE_PATH, matrix_88, keys, calls, feature_names_88,
        source_provenance=source_provenance,
    )
    del egemaps_data, matrix_88
    egemaps_data = load_egemaps_cache(
        EGEMAPS_CACHE_PATH, keys, calls,
        source_provenance=source_provenance,
    )
print("Validated cache:", EGEMAPS_CACHE_PATH)
print("Shape / dtype:", egemaps_data.X.shape, egemaps_data.X.dtype)
display(pd.DataFrame([dict(egemaps_data.metadata)]))


## Checkpoint E — 두 representation의 공정 비교 guard

두 입력의 행·label·call·split과 M04-100 설정을 fit 전에 확인한다. scaler는 representation마다 같은 internal Training 행에서만 fit한다. 평가는 두 결과 모두 기존 calculate_prediction_metrics를 사용한다.


In [ ]:
expected_m04 = {**dict(M04_MLP_CONFIG), "max_iter": 100}
checks = {
    "record_key": np.array_equal(egemaps_data.record_keys, keys),
    "labels_59d_train": np.array_equal(prepared_59d.y_train, labels[train_idx]),
    "labels_59d_validation": np.array_equal(
        prepared_59d.y_evaluation, labels[validation_idx]
    ),
    "call_id": np.array_equal(egemaps_data.call_ids, calls),
    "train_indices": np.array_equal(prepared_59d.train_idx, train_idx),
    "validation_indices": np.array_equal(
        prepared_59d.validation_idx, validation_idx
    ),
    "seed": split_audit["seed"] == expected_m04["random_state"] == 42,
    "model_config": dict(M04_ITER100_FEATURE_SCREENING_CONFIG) == expected_m04,
    "59d_scaler_fit_rows": int(prepared_59d.scaler.n_samples_seen_) == len(train_idx),
    "88d_feature_order": egemaps_data.feature_names == feature_names_88,
    "88d_shape_dtype_finite": (
        egemaps_data.X.shape == (len(keys), 88)
        and egemaps_data.X.dtype == np.float32
        and np.isfinite(egemaps_data.X).all()
    ),
    "split_audit": dict(prepared_59d.split_audit) == split_audit,
}
display(pd.DataFrame([
    {"contract": name, "pass": bool(passed)}
    for name, passed in checks.items()
]))
if not all(checks.values()):
    raise RuntimeError("STOP: fair comparison input contract differs")


## Checkpoint F — 59D M04-100 internal reference

이 셀을 직접 실행하면 59D를 기존 M04-100 runner로 fit한다. 저장된 과거 Accuracy를 현재 결과로 재사용하지 않는다.


In [ ]:
result_59d = _run_prepared_representation(
    prepared_59d, experiment_id="AF10_AF11_COMBINATION", allow_fit=True
)
if dict(result_59d.model_config) != expected_m04:
    raise RuntimeError("STOP: 59D M04-100 configuration changed")
if result_59d.fit_audit["scaler_fit_scope"] != "internal_train_only":
    raise RuntimeError("STOP: 59D scaler fit scope changed")
if not np.array_equal(result_59d.fit_audit["train_idx"], train_idx):
    raise RuntimeError("STOP: 59D train indices changed")
train_metrics_59d = calculate_prediction_metrics(
    prepared_59d.y_train,
    result_59d.model.predict(prepared_59d.X_train_scaled),
)
report_59d = {
    "representation": "AF10_AF11_COMBINATION",
    "dimension": 59,
    "train_accuracy": train_metrics_59d["accuracy"],
    "internal_validation_accuracy": result_59d.metrics["accuracy"],
    "generalization_gap": (
        train_metrics_59d["accuracy"] - result_59d.metrics["accuracy"]
    ),
}
display(pd.DataFrame([report_59d]))


## Checkpoint G — eGeMAPSv02 88D, 동일 M04-100

기존 handcrafted 준비 helper는 새 feature 이름을 의도적으로 거부한다. 여기서는 같은 StandardScaler를 같은 internal Training index에 fit해 기존 PreparedRepresentation에 담고, F와 동일한 runner를 호출한다. 이 셀을 직접 실행할 때만 88D 모델을 fit한다.


In [ ]:
scaler_88 = StandardScaler()
X_train_88 = scaler_88.fit_transform(
    egemaps_data.X[train_idx]
).astype(np.float32, copy=False)
X_validation_88 = scaler_88.transform(
    egemaps_data.X[validation_idx]
).astype(np.float32, copy=False)
if int(scaler_88.n_samples_seen_) != len(train_idx):
    raise RuntimeError("STOP: 88D scaler was not fit on internal Training only")
if not np.isfinite(X_train_88).all() or not np.isfinite(X_validation_88).all():
    raise ValueError("STOP: scaled 88D matrix contains NaN or inf")
prepared_88d = PreparedRepresentation(
    representation_id=REPRESENTATION_ID,
    feature_names=egemaps_data.feature_names,
    X_train_scaled=X_train_88,
    X_evaluation_scaled=X_validation_88,
    y_train=labels[train_idx],
    y_evaluation=labels[validation_idx],
    training_record_keys=keys[train_idx],
    evaluation_record_keys=keys[validation_idx],
    train_idx=train_idx.copy(),
    validation_idx=validation_idx.copy(),
    scaler=scaler_88,
    split_audit=split_audit,
)
result_88d = _run_prepared_representation(
    prepared_88d, experiment_id=REPRESENTATION_ID, allow_fit=True
)
if dict(result_88d.model_config) != expected_m04:
    raise RuntimeError("STOP: eGeMAPS M04-100 configuration changed")
if result_88d.fit_audit["scaler_fit_scope"] != "internal_train_only":
    raise RuntimeError("STOP: 88D scaler fit scope changed")
train_metrics_88d = calculate_prediction_metrics(
    prepared_88d.y_train,
    result_88d.model.predict(prepared_88d.X_train_scaled),
)
report_88d = {
    "representation": REPRESENTATION_ID,
    "dimension": 88,
    "train_accuracy": train_metrics_88d["accuracy"],
    "internal_validation_accuracy": result_88d.metrics["accuracy"],
    "generalization_gap": (
        train_metrics_88d["accuracy"] - result_88d.metrics["accuracy"]
    ),
}
display(pd.DataFrame([report_88d]))


## Checkpoint H — paired fair comparison

fit audit를 다시 확인한 후 같은 validation 행의 prediction을 짝지어 비교한다. Accuracy 차이와 오류 건수는 관찰값이며 원인이나 우열 판단은 여기서 하지 않는다.


In [ ]:
for label, result in (("59D", result_59d), ("eGeMAPS", result_88d)):
    audit = result.fit_audit
    if (
        not np.array_equal(audit["train_idx"], train_idx)
        or not np.array_equal(audit["validation_idx"], validation_idx)
        or not np.array_equal(audit["training_record_keys"], keys[train_idx])
        or not np.array_equal(audit["validation_record_keys"], keys[validation_idx])
        or not np.array_equal(result.y_true, labels[validation_idx])
        or audit["seed"] != 42
        or audit["scaler_fit_scope"] != "internal_train_only"
        or dict(result.model_config) != expected_m04
    ):
        raise RuntimeError(f"STOP: {label} fit/audit comparison contract differs")
if not np.array_equal(result_59d.y_true, result_88d.y_true):
    raise RuntimeError("STOP: validation labels differ")
paired = calculate_paired_prediction_changes(
    y_true=labels[validation_idx],
    m00_pred=result_59d.y_pred,
    m01_pred=result_88d.y_pred,
    m00_record_keys=result_59d.fit_audit["validation_record_keys"],
    m01_record_keys=result_88d.fit_audit["validation_record_keys"],
)
paired_summary = {
    "both_correct": paired["both_correct"],
    "both_wrong": paired["both_wrong"],
    "59D_wrong_to_eGeMAPS_correct": paired["m00_wrong_to_m01_correct"],
    "59D_correct_to_eGeMAPS_wrong": paired["m00_correct_to_m01_wrong"],
    "net_correct_gain": (
        paired["m00_wrong_to_m01_correct"]
        - paired["m00_correct_to_m01_wrong"]
    ),
}
comparison_metrics = pd.DataFrame([
    {
        "representation": label,
        "accuracy": result.metrics["accuracy"],
        "confusion_matrix": result.metrics["confusion_matrix"],
        "recall_speaker_0": result.metrics["recall_speaker_0"],
        "recall_speaker_1": result.metrics["recall_speaker_1"],
    }
    for label, result in (("59D", result_59d), ("eGeMAPS", result_88d))
])
display(comparison_metrics)
display(pd.DataFrame([paired_summary]))
print("Internal Validation Accuracy delta:",
      result_88d.metrics["accuracy"] - result_59d.metrics["accuracy"])


## Checkpoint I — duration·overlap·confidence 진단

기존 진단 helper를 두 결과에 똑같이 적용한다. duration, absolute crop 위치와 overlap은 이 셀의 진단용 tracking 정보이며 모델 feature가 아니다. 각 모델의 confidence는 calibration이 다를 수 있으므로 직접 비교할 때 주의한다.


In [ ]:
training_tracking = e05_cache.tracking.iloc[train_idx].reset_index(drop=True)
validation_tracking = e05_cache.tracking.iloc[validation_idx].reset_index(drop=True)
diagnostics = {
    name: build_standalone_diagnostic_report(
        result,
        training_tracking=training_tracking,
        evaluation_tracking=validation_tracking,
    )
    for name, result in (("59D", result_59d), ("eGeMAPS", result_88d))
}
for slice_name in ("duration_summary", "overlap_summary", "confidence_summary"):
    print(slice_name)
    display(pd.concat([
        diagnostics[name][slice_name].assign(representation=name)
        for name in ("59D", "eGeMAPS")
    ], ignore_index=True))
display(pd.DataFrame([
    {
        "representation": name,
        "high_confidence_error_count":
            diagnostics[name]["high_confidence_error_count"],
    }
    for name in ("59D", "eGeMAPS")
]))


## Checkpoint J — 결과 표와 사용자 해석

관찰값만 출력한다. 아래 빈 항목은 사용자가 실행 결과를 보고 직접 작성한다.


In [ ]:
summary_table = pd.DataFrame([report_59d, report_88d])
display(summary_table)
display(comparison_metrics)
display(pd.DataFrame([paired_summary]))
print("Cache:", EGEMAPS_CACHE_PATH)
print("Cache bytes:", EGEMAPS_CACHE_PATH.stat().st_size)
print("Result interpretation: write only after reviewing the observed outputs.")


### 사용자 해석 메모

- 관찰:
- 59D 대비 eGeMAPS 변화:
- class별 변화:
- 주요 error 변화:
- generalization gap 변화:
- 비용 변화:
- leakage / comparison integrity 확인:
- Stage 5로 가져갈 가설:
- 다음 판단:
